# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shravdrao/FLYRANK-/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [13]:
"One row = one webpage (content) for one day."

'One row = one webpage (content) for one day.'

Because we have:

1.content_hash_id → identifies the webpage.
2.report_date → identifies the day.

That means:

One row = one webpage on one specific date.

In [10]:
!pip install -q duckdb

In [11]:
import duckdb
from google.colab import userdata

# Read the secret from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Register your Hugging Face token
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

warehouse = "hf://datasets/FlyRank/internship-warehouse"

print("Connected successfully!")

Connected successfully!


In [12]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{warehouse}/fact_content_daily_performance/**/*.parquet')
LIMIT 5
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:**

1.gsc_impressions

2.gsc_clicks

3.gsc_avg_position

4.ga4_engagement_rate

5.word_count




**Label / Proxy**

1.Context

2.content_type

3.main_intent

4.report_date

5.month

**Excluded**

1.Future outcome fields (e.g., future trend/label columns) → Excluded because they cause data leakage.

2.client_hash_id → Excluded to protect client privacy.

3.content_hash_id → Excluded because it is only an identifier, not a useful feature.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**1.Verify the Grain (Unit of Analysis) **

This query verifies that each row represents one content page on one reporting date.

In [14]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (content_hash_id, report_date)) AS unique_content_day
FROM read_parquet('{warehouse}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ unique_content_day │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            9841378 │
└────────────┴────────────────────┘



**2.Verify Row Count and Time Window**

This query confirms the number of rows and that the data covers the March 2026 time window.

In [15]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{warehouse}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
""").show()

┌────────────┬────────────┬────────────┐
│ total_rows │ start_date │  end_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘



**3.Verify Data Availability**
This query verifies how many rows have both Google Search Console and Google Analytics data available for analysis.

In [8]:
import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")

print("Secret created successfully!")

con.sql("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5;
""").show()

Secret created successfully!
┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ a

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [9]:
"Data Limits: This dataset cannot explain why Google ranks pages the way it does. Some rows have missing GA4 data, earlier records may only contain GSC data, and different reporting windows may overlap. Because of these limitations, the results should be used for observing patterns and supporting SEO decisions, not for proving cause and effect."

'Data Limits: This dataset cannot explain why Google ranks pages the way it does. Some rows have missing GA4 data, earlier records may only contain GSC data, and different reporting windows may overlap. Because of these limitations, the results should be used for observing patterns and supporting SEO decisions, not for proving cause and effect.'

## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.